In [71]:
from sklearn.linear_model import LassoCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import seaborn as sns

In [72]:
live = pd.read_csv("../data/samples/trial/live_metrics.csv")
verbose = pd.read_csv("../data/samples/trial/verbose_statements.csv")
initial = pd.read_csv("../data/samples/trial/initial_statement.csv")

In [73]:
live = live.drop(['Unnamed: 0', "_id", "Category", "Collect", "Current Time"], axis=1)

In [74]:
response_time = verbose['total_duration'].tolist()
response_time = response_time[:-1]
reset_iters = live[live['Iteration'] == 1].index.tolist()

gpu_util_avg = []
memory_util_avg = []
clock_util_avg = []

for i in range(1, len(reset_iters)):
    temp_metrics = live.dropna()
    gpu_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 3].tolist()
    memeory_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 9].tolist()
    clock_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], -2].tolist()
    gpu_util_avg.append(np.average(gpu_util_prompt))
    memory_util_avg.append(np.average(memeory_util_prompt))
    clock_util_avg.append(np.average(clock_util_prompt))

In [ ]:
live = live.dropna()
live.corr()

In [76]:
live = live.drop(columns=['Memory Clock Utilization', 'Memory Current Clock (MHz)'])

In [ ]:
mask = np.triu(np.ones_like(live.corr(), dtype=bool))
heatmap = sns.heatmap(live.corr(), mask=mask, annot=True, cmap="crest")

In [ ]:
sns.set_theme()
g = sns.PairGrid(live.iloc[:, 4:])
plot = g.map(sns.scatterplot)
# sns.pairplot(live.iloc[:, 4:], corner=True)

In [79]:
X = pd.DataFrame([gpu_util_avg, memory_util_avg, clock_util_avg])
X = X.T
X.columns = ['GPU Util', "Memory Util", "GPU Clock"]
X_sam = live
y = pd.DataFrame(response_time, columns=['AVG Response Time'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_sam, y, test_size=0.2, shuffle=True, random_state=20)

In [102]:
empty_df = {"index":range(70)}
live_avg = pd.DataFrame(empty_df)

In [105]:
live_avg.insert(column='Column', value=10)

TypeError: DataFrame.insert() missing 1 required positional argument: 'loc'

In [ ]:
# column_names = live.columns.tolist()
# live_avg = pd.DataFrame(data=[range(70)], columns=column_names)
# # live_avg.index = [range(70)]


empty_df = {"index":range(70)}
live_avg = pd.DataFrame(empty_df)


for iter in range(1, len(reset_iters)):
    for column in live.columns:
        temp_list = live.iloc[reset_iters[iter-1]:reset_iters[iter],  column_names.index(column)].tolist()
        live_avg.iloc[iter-1, column_names.index(column)] = np.average(temp_list)

ValueError: 9 columns passed, passed data had 70 columns

In [92]:
live_avg

,GPU Utilization (%),Power Draw (Watts),GPU Temp (°C),GPU Current Clock (MHz),Memory Allocation Used (MB),Memory Utilization (%),Time Delta,Iteration,GPU Clock Utilization


In [ ]:
lcv = LassoCV(cv=3)
lcv.fit(X_train, y_train)
score = lcv.score(X_test, y_test)
print(f"Accuracy: {score}")

Accuracy: 0.19863702337594102


c:\Users\rahul\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:1656: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [ ]:
lcv.coef_

array([  87.23131392, -118.20318102,    1.38325616])